## Vector DB

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.06.23 </div>
<div style="text-align: right"> last update : 2026.06.23 </div>

개정 이력  
- `2026.06.23` : 노트북 최초 작성

### 벡터 DB란?

**벡터 DB(Vector Database)** 는 텍스트를 **숫자 벡터(embedding)** 로 바꿔 저장해 두고, *의미가 비슷한* 내용을 빠르게 찾아주는 데이터베이스입니다.

키워드가 정확히 일치하지 않아도(예: "중대재해"로 검색했는데 "사망 등 재해 정도가 심한 경우"라는 문장을 찾아냄) **뜻이 가까운** 문장을 꺼내올 수 있다는 점이 핵심입니다.

이 노트북에서는 RAG(검색 증강 생성)의 **사전작업(Pre-processing)** 을 다룹니다.

> 1. **문서 로드(Load)** : PDF에서 텍스트를 읽어온다
> 2. **분할(Split)** : 문서를 적당한 크기의 조각(chunk)으로 나눈다
> 3. **임베딩(Embedding)** : 각 조각을 벡터로 변환한다
> 4. **벡터 DB 저장(Store)** : 벡터를 FAISS 인덱스에 저장한다

마지막에 만든 인덱스를 디스크에 저장해 두고, **다음 노트북 `2.2_rag.ipynb` 에서 불러와** 질문에 답하는 RAG를 완성합니다.

실습 문서는 `data/osh_act.pdf` (**산업안전보건법**) 입니다.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# 사용할 임베딩 모델 확인 (.env 의 VOYAGE_EMBEDDING_MODEL)
print(os.getenv("VOYAGE_EMBEDDING_MODEL"))

voyage-4-lite


### 1. 문서 로드 (Document Load)

가장 먼저 PDF 파일에서 텍스트를 읽어옵니다. 여기서는 `PyMuPDFLoader` 를 사용합니다.

로더는 PDF를 읽어 **페이지 단위의 `Document` 객체 리스트**로 돌려줍니다. 각 `Document` 는 본문(`page_content`)과 출처·페이지 번호 등의 정보(`metadata`)를 담고 있습니다.

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../data/osh_act.pdf")
docs = loader.load()
docs = docs[:20]  # 실습용: 무료 임베딩 한도(3 RPM / 10K TPM)에 맞춰 앞 20페이지만 사용

print(f"문서의 페이지 수: {len(docs)}")

문서의 페이지 수: 20


한 페이지의 본문 내용을 확인해 봅니다.

In [4]:
print(docs[0].page_content)

법제처                                                            1                                                       국가법령정보센터
산업안전보건법
 
산업안전보건법
[시행 2026. 6. 1.] [법률 제21374호, 2026. 2. 19., 일부개정]
고용노동부 (산업안전보건정책과-과태료, 적용범위, 공표, 지자체·이사회보고) 044-202-8810, 8813, 8815, 8997
고용노동부 (산업안전기준과-도급안전조치, 인증검사, 안전관리자-제조업) 044-202-8855, 8857, 8856
고용노동부 (산업보건기준과-감염병·석면, 건강진단, 작업환경측정·유해물질, 보건관리자·보건조치) 044-202-8876, 8874, 8878, 8875
고용노동부 (직업건강증진팀-휴게시설·고객응대, 미세먼지·고열, 직무스트레스·과로) 044-202-8893, 8895, 8894
고용노동부 (안전보건감독기획과-산재발생보고) 044-202-8910
고용노동부 (건설산재예방과-안전관리비·재해예방지도기관, 환산재해율, 안전관리자-건설업) 044-202-8942, 8939, 8940
고용노동부 (안전문화협력팀-안전보건교육) 044-202-8820, 8993, 8921
고용노동부 (산업안전보건정책과-산보위·관리책임자·관리감독자) 044-202-8996
고용노동부 (화학사고예방과-MSDS, PSM) 044-202-8971, 8967
       제1장 총칙
 
제1조(목적) 이 법은 산업 안전 및 보건에 관한 기준을 확립하고 그 책임의 소재를 명확하게 하여 산업재해를 예방하고
쾌적한 작업환경을 조성함으로써 노무를 제공하는 사람의 안전 및 보건을 유지ㆍ증진함을 목적으로 한다. <개정
2020. 5. 26.>
 
제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2020. 5. 26., 2023. 8. 8.>
1. “산업재해”란 노무를 제공하는 사람이 업무에 관계되는 건

본문과 함께 따라오는 **메타데이터** 도 살펴봅니다. 나중에 RAG 답변의 **출처(몇 페이지에서 가져왔는지)** 를 밝힐 때 이 정보를 사용합니다.

In [5]:
docs[1].metadata

{'producer': 'iText 2.1.7 by 1T3XT',
 'creator': '',
 'creationdate': '2026-06-23T12:59:28+09:00',
 'source': '../data/osh_act.pdf',
 'file_path': '../data/osh_act.pdf',
 'total_pages': 52,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2026-06-23T12:59:28+09:00',
 'trapped': '',
 'modDate': "D:20260623125928+09'00'",
 'creationDate': "D:20260623125928+09'00'",
 'page': 1}

### 2. 문서 분할 (Text Split)

PDF 한 페이지는 너무 길어서 그대로 검색에 쓰기 어렵습니다. 긴 글을 통째로 임베딩하면 "이 페이지는 대략 이런 내용"이라는 **뭉뚱그려진 벡터** 가 되어, 정작 필요한 한 문장을 정확히 찾기 어려워집니다.

그래서 문서를 **작은 조각(chunk)** 으로 나눕니다.

- **`chunk_size`** : 조각 하나의 최대 글자 수
- **`chunk_overlap`** : 인접한 조각끼리 겹치는 글자 수. 조각 경계에서 문맥이 끊기는 것을 막아 줍니다. (예: 한 조항이 두 조각으로 잘려도 겹치는 부분 덕분에 의미가 이어짐)

`RecursiveCharacterTextSplitter` 는 문단 → 문장 → 단어 순으로 **자연스러운 경계** 를 우선해 잘라 줍니다.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100)
split_documents = text_splitter.split_documents(docs)

print(f"분할된 청크의 수: {len(split_documents)}")

분할된 청크의 수: 28


분할된 청크 하나를 확인해 봅니다. 페이지 전체보다 훨씬 짧고 다루기 쉬운 단위가 되었습니다.

In [7]:
print(split_documents[10].page_content)

법제처                                                            10                                                       국가법령정보센터
산업안전보건법
제36조(위험성평가의 실시) ① 사업주는 건설물, 기계ㆍ기구ㆍ설비, 원재료, 가스, 증기, 분진, 근로자의 작업행동 또는
그 밖의 업무로 인한 유해ㆍ위험 요인을 찾아내어 사망, 부상 및 질병으로 이어질 수 있는 위험성의 크기가 허용 가
능한 수준인지를 결정하고, 그 위험성을 줄이기 위하여 이 법과 이 법에 따른 명령에 따른 조치를 포함하여 근로자에
대한 위험 또는 건강장해를 방지하기 위한 개선대책을 수립하고 이행(이하 “위험성평가”라 한다) 하여야 한다. <개정
2026. 2. 19.>
② 사업주는 위험성평가 시 고용노동부령으로 정하는 바에 따라 위험성평가에 해당 사업장의 근로자를 참여시켜야
한다.<개정 2026. 2. 19.>
③ 사업주는 근로자대표가 요구하면 위험성평가 시 근로자대표를 참여시켜야 한다.<신설 2026. 2. 19.>
④ 사업주는 고용노동부령으로 정하는 위험성평가 관련 사항을 해당 사업장의 근로자에게 제29조에 따른 안전보건
교육, 설명회, 사업장 게시, 서면 또는 전자적 방법 등으로 알려야 한다. 이 경우 사업주는 중대재해로 이어질 수 있
는 유해ㆍ위험 요인에 대하여 작업 전 안전점검회의 등을 통하여 근로자에게 상시적으로 주지시키도록 노력하여야
한다.<신설 2026. 2. 19.>
⑤ 사업주는 위험성평가의 결과를 고용노동부령으로 정하는 바에 따라 기록하여 보존하여야 한다.<개정 2026. 2.
19.>
⑥ 위험성평가의 방법, 절차 및 시기, 그 밖에 필요한 사항은 고용노동부령으로 정한다.<개정 2026. 2. 19.>
 
제37조(안전보건표지의 설치ㆍ부착) ① 사업주는 유해하거나 위험한 장소ㆍ시설ㆍ물질에 대한 경고, 비상시에 대처하
기 위한 지시ㆍ안내 또는 그 밖에 근로자의 안전 및 보건 의식을 고취

### 3. 임베딩 (Embedding)

**임베딩** 은 텍스트를 *의미를 담은 숫자 벡터* 로 바꾸는 작업입니다. 비슷한 뜻의 문장은 벡터 공간에서 가까운 위치에 놓이기 때문에, 이 벡터들의 거리를 비교해 "의미가 가까운 문장"을 찾을 수 있습니다.

여기서는 **Voyage AI** 의 임베딩 모델을 사용합니다. 모델 이름은 `.env` 의 `VOYAGE_EMBEDDING_MODEL` 에서 읽어옵니다.

In [8]:
import time

from langchain_voyageai import VoyageAIEmbeddings


class RateLimitedVoyageEmbeddings(VoyageAIEmbeddings):
    """무료 한도(3 RPM / 10K TPM) 초과 시 잠시 대기 후 자동 재시도하는 임베딩.

    RateLimitError 가 나면 25초 기다렸다가 다시 시도하므로,
    빌드·검색 어디서 호출되든 한도에 걸려도 알아서 통과한다.
    (근본 해결은 결제수단 등록 — 무료 토큰은 그대로 적용된다.)
    """

    def _with_retry(self, fn, arg):
        for attempt in range(6):
            try:
                return fn(arg)
            except Exception as e:
                if "RateLimit" not in type(e).__name__:
                    raise
                print(f"  [rate limit] 25초 대기 후 재시도 ({attempt + 1}/6)")
                time.sleep(25)
        raise RuntimeError("재시도 초과 - 잠시 후 다시 실행하거나 결제수단을 등록하세요.")

    def embed_documents(self, texts):
        return self._with_retry(super().embed_documents, texts)

    def embed_query(self, text):
        return self._with_retry(super().embed_query, text)


embeddings = RateLimitedVoyageEmbeddings(model=os.getenv("VOYAGE_EMBEDDING_MODEL"))

임베딩이 실제로 어떤 결과를 내는지 감을 잡기 위해, 짧은 문장 하나를 벡터로 바꿔 **벡터의 차원(숫자 개수)** 을 확인해 봅니다. 텍스트가 수백 개의 숫자로 이루어진 벡터로 표현되는 것을 볼 수 있습니다.

In [9]:
sample_vector = embeddings.embed_query("중대재해란 무엇인가?")

print(f"벡터 차원: {len(sample_vector)}")
print(f"앞부분 5개 값: {sample_vector[:5]}")

벡터 차원: 1024
앞부분 5개 값: [-0.04009825736284256, -0.023908350616693497, 0.007812571711838245, 0.04819321259856224, -0.04292207956314087]


### 4. 벡터 DB 생성 (Create Vector Store / FAISS)

**FAISS(Facebook AI Similarity Search)** 는 벡터들을 저장하고 *가장 가까운 벡터* 를 빠르게 찾아주는 라이브러리입니다. 별도 서버 없이 로컬에서 동작하므로 실습에 적합합니다.

분할된 청크를 **임베딩(3단계)** 으로 벡터화해 인덱스에 담습니다.

> ⚠️ **Voyage 무료 한도(3 RPM / 10K TPM)**
> 20페이지만 써도 총 토큰이 약 2만으로 분당 한도(1만)의 2배입니다. 그래서 `FAISS.from_documents` 로 한꺼번에 보내면 한도를 초과해 `RateLimitError` 가 납니다.
> 아래처럼 **청크를 조금씩 나눠 보내고, 요청 사이에 잠시 대기(throttle)** 하면 무료 한도 안에서 인덱스를 만들 수 있습니다. (약 4~5분 소요)
> 결제수단을 등록하면 한도가 올라가 `FAISS.from_documents` 한 줄로 끝납니다(무료 토큰은 그대로 적용).

In [10]:
import time

from langchain_community.vectorstores import FAISS

# Voyage 무료 한도(3 RPM / 10K TPM)에 맞춰 청크를 나눠 쉬어가며 임베딩한다.
BATCH = 6   # 한 요청에 보낼 청크 수 (한 요청이 10K 토큰을 넘지 않도록 작게)
SLEEP = 60  # 요청 간 대기(초). 분당 1요청 → RPM·TPM 모두 안전

vectorstore = None
total = len(split_documents)

for i in range(0, total, BATCH):
    batch_docs = split_documents[i : i + BATCH]

    # 한도 초과 시 잠시 더 기다렸다가 재시도
    for attempt in range(5):
        try:
            if vectorstore is None:
                vectorstore = FAISS.from_documents(batch_docs, embeddings)
            else:
                vectorstore.add_documents(batch_docs)
            break
        except Exception as e:
            print(f"  재시도 {attempt + 1}/5 - 30초 대기 ({type(e).__name__})")
            time.sleep(30)
    else:
        raise RuntimeError("임베딩 반복 실패 - BATCH를 줄이거나 결제수단을 등록하세요.")

    done = min(i + BATCH, total)
    print(f"임베딩 진행: {done}/{total}")
    if done < total:
        time.sleep(SLEEP)

print("벡터 DB 생성 완료")

임베딩 진행: 6/28
임베딩 진행: 12/28
임베딩 진행: 18/28
임베딩 진행: 24/28
임베딩 진행: 28/28
벡터 DB 생성 완료


### 5. 유사도 검색 (Similarity Search)

이제 벡터 DB에 질문을 던져 **의미가 가까운 청크** 를 찾아봅니다. 질문도 내부적으로 임베딩된 뒤, 저장된 청크 벡터들과 거리를 비교합니다.

In [11]:
results = vectorstore.similarity_search("중대재해란 무엇인가?")

for doc in results:
    print(doc.page_content)
    print("-" * 60)

등”이라 한다)가 발생하였을 때에는 그 원인 규명 또는 산업재해 예방대책 수립을 위하여 그 발생 원인을 조사할 수
있다. <개정 2026. 2. 19.>
1. 중대재해
2. 화재ㆍ폭발, 붕괴 등으로 산업재해(중대재해는 제외한다)가 발생하여 중대재해 예방을 위하여 원인조사가 필요하
다고 인정되는 경우로서 고용노동부령으로 정하는 산업재해
------------------------------------------------------------
법제처                                                            14                                                       국가법령정보센터
산업안전보건법
③ 관리감독자등은 제2항에 따른 보고를 받으면 안전 및 보건에 관하여 필요한 조치를 하여야 한다.
④ 사업주는 산업재해가 발생할 급박한 위험이 있다고 근로자가 믿을 만한 합리적인 이유가 있을 때에는 제1항에
따라 작업을 중지하고 대피한 근로자에 대하여 해고나 그 밖의 불리한 처우를 해서는 아니 된다.
 
제53조(고용노동부장관의 시정조치 등) ① 고용노동부장관은 사업주가 사업장의 건설물 또는 그 부속건설물 및 기계ㆍ
기구ㆍ설비ㆍ원재료(이하 “기계ㆍ설비등”이라 한다)에 대하여 안전 및 보건에 관하여 고용노동부령으로 정하는 필요
한 조치를 하지 아니하여 근로자에게 현저한 유해ㆍ위험이 초래될 우려가 있다고 판단될 때에는 해당 기계ㆍ설비등
에 대하여 사용중지ㆍ대체ㆍ제거 또는 시설의 개선, 그 밖에 안전 및 보건에 관하여 고용노동부령으로 정하는 필요
한 조치(이하 “시정조치”라 한다)를 명할 수 있다.
② 제1항에 따라 시정조치 명령을 받은 사업주는 해당 기계ㆍ설비등에 대하여 시정조치를 완료할 때까지 시정조치
명령 사항을 사업장 내에 근로자가 쉽게 볼 수 있는 장소에 게시하여야 한다.
③ 고용노동부장관은 사업주가 해당 기계ㆍ설비등에 대한 시정조치 명령을 이행하지 아니하여 유해ㆍ위험 상태가
해

`similarity_search_with_score` 를 쓰면 각 결과의 **거리 점수** 도 함께 볼 수 있습니다. 이 점수는 **작을수록 질문과 더 가깝다(유사하다)** 는 뜻입니다.

In [12]:
results_with_score = vectorstore.similarity_search_with_score("중대재해란 무엇인가?", k=3)

for doc, score in results_with_score:
    print(f"[점수: {score:.4f}]")
    print(doc.page_content[:150], "...")
    print("-" * 60)

[점수: 1.0376]
등”이라 한다)가 발생하였을 때에는 그 원인 규명 또는 산업재해 예방대책 수립을 위하여 그 발생 원인을 조사할 수
있다. <개정 2026. 2. 19.>
1. 중대재해
2. 화재ㆍ폭발, 붕괴 등으로 산업재해(중대재해는 제외한다)가 발생하여 중대재해 예방을 위하여 원인조 ...
------------------------------------------------------------
[점수: 1.1556]
법제처                                                            14                                                       국가법령정보센터
산업안전보건법
③ 관리감독자등은 제2항 ...
------------------------------------------------------------
[점수: 1.3121]
법제처                                                            15                                                       국가법령정보센터
산업안전보건법
② 고용노동부장관은 제1 ...
------------------------------------------------------------


실무에서는 보통 벡터 DB를 직접 호출하기보다 **검색기(Retriever)** 형태로 감싸 사용합니다. 검색기는 "질문을 넣으면 관련 문서를 돌려주는" 표준 인터페이스로, 뒤 노트북에서 RAG 체인에 그대로 연결됩니다.

- `search_kwargs={"k": 3}` : 가장 가까운 청크를 **몇 개** 가져올지 지정 (여기서는 3개)

In [13]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retriever.invoke("근로자의 의무는 무엇인가?")

[Document(id='319da9d8-e631-4d57-a48e-5c6b01a4eff0', metadata={'producer': 'iText 2.1.7 by 1T3XT', 'creator': '', 'creationdate': '2026-06-23T12:59:28+09:00', 'source': '../data/osh_act.pdf', 'file_path': '../data/osh_act.pdf', 'total_pages': 52, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-23T12:59:28+09:00', 'trapped': '', 'modDate': "D:20260623125928+09'00'", 'creationDate': "D:20260623125928+09'00'", 'page': 12}, page_content='작업장소에서 대피시키는 등 안전 및 보건에 관하여 필요한 조치를 하여야 한다.\n \n제52조(근로자의 작업중지) ① 근로자는 산업재해가 발생할 급박한 위험이 있는 경우에는 작업을 중지하고 대피할 수\n있다.\n② 제1항에 따라 작업을 중지하고 대피한 근로자는 지체 없이 그 사실을 관리감독자 또는 그 밖에 부서의 장(이하\n“관리감독자등”이라 한다)에게 보고하여야 한다.'),
 Document(id='90844ce5-9ffe-4f36-aae6-d7979b6a2344', metadata={'producer': 'iText 2.1.7 by 1T3XT', 'creator': '', 'creationdate': '2026-06-23T12:59:28+09:00', 'source': '../data/osh_act.pdf', 'file_path': '../data/osh_act.pdf', 'total_pages': 52, 'format': 'PDF 1.4', 'title': '', 'author': '', 'sub

법령 문서에 어울리는 다른 질문들도 검색해 봅니다.

In [14]:
for question in ["산업재해의 정의", "도급인이란", "사업주의 의무"]:
    print(f"질문: {question}")
    top = retriever.invoke(question)[0]
    print(f"  └ {top.page_content[:120]} ...\n")

질문: 산업재해의 정의
  └ 법제처                                                            1                                                       국 ...

질문: 도급인이란
  └ 법제처                                                            16                                                        ...

질문: 사업주의 의무
  [rate limit] 25초 대기 후 재시도 (1/6)
  [rate limit] 25초 대기 후 재시도 (2/6)
  [rate limit] 25초 대기 후 재시도 (3/6)
  └ 법제처                                                            2                                                       국 ...



### 6. 인덱스 저장 (Save Index)

만든 벡터 DB를 디스크에 저장합니다. 매번 PDF를 다시 읽고 임베딩하는 **비용·시간을 아끼기 위해서** 입니다.

`save_local` 은 지정한 폴더에 두 개의 파일을 만듭니다.
- `index.faiss` : 벡터 인덱스 본체
- `index.pkl` : 청크 본문·메타데이터

다음 노트북 `2.2_rag.ipynb` 에서 `FAISS.load_local` 로 이 폴더를 불러와 RAG를 수행합니다.

In [15]:
INDEX_PATH = "../data/vector_db/osh_act_faiss"

vectorstore.save_local(INDEX_PATH)
print(f"저장 완료: {INDEX_PATH}")
print(os.listdir(INDEX_PATH))

저장 완료: ../data/vector_db/osh_act_faiss
['index.faiss', 'index.pkl']


### 전체 파이프라인 한눈에 보기

지금까지의 사전작업(로드 → 분할 → 임베딩 → 저장)을 한 셀로 압축하면 다음과 같습니다.

In [16]:
# # 전체 파이프라인 요약 (참고용 — 실제 실행은 위 셀들을 사용)
# import time
# from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_voyageai import VoyageAIEmbeddings
# from langchain_community.vectorstores import FAISS
#
# # 1. 문서 로드 (실습용: 앞 20페이지)
# docs = PyMuPDFLoader("../data/osh_act.pdf").load()[:20]
#
# # 2. 문서 분할
# split_documents = RecursiveCharacterTextSplitter(
#     chunk_size=2000, chunk_overlap=100
# ).split_documents(docs)
#
# # 3. 임베딩
# embeddings = VoyageAIEmbeddings(model=os.getenv("VOYAGE_EMBEDDING_MODEL"))
#
# # 4. 벡터 DB 생성 (무료 한도 시 위 셀처럼 batch + sleep 으로 나눠 호출)
# vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)
#
# # 5. 저장
# vectorstore.save_local("../data/vector_db/osh_act_faiss")
# print("벡터 DB 생성 및 저장 완료")

### 정리

- **벡터 DB** 는 텍스트를 임베딩(벡터)으로 저장해 **의미 기반 검색** 을 가능하게 한다.
- RAG 사전작업은 **로드 → 분할 → 임베딩 → 저장** 의 4단계로 이뤄진다.
- `chunk_size`/`chunk_overlap` 은 검색 품질에 직접 영향을 준다. 문서 성격에 맞게 조정한다.
- 만든 인덱스를 `save_local` 로 저장하면 재사용할 수 있다.

> 다음 노트북 `2.2_rag.ipynb` 에서는 이 인덱스를 불러와 **프롬프트 + LLM** 과 연결해 질문에 답하는 RAG를 완성합니다.